# EEG Motor Imagery Classification v8 — Subject-Aligned, Trial-Level, Leak-Free

**Same leak-free evaluation as v6** (GroupKFold, every supervised step fitted on training subjects only), plus:

1. **Per-subject z-normalization** — removes inter-subject amplitude/impedance differences (label-free)
2. **Filter Bank CSP (FBCSP)** — CSP in 7 frequency sub-bands (BCI Competition IV winning method)
3. **Spectrogram features (STFT)** — band power over time, not just averaged
4. **Scalogram features (CWT, Morlet)** — multi-resolution time-frequency energy
5. **Connectivity features** — left/right hemisphere coherence and power asymmetry
6. **EEGNet** — compact CNN designed for EEG BCIs
7. **Hierarchical 12-class** — Action/Imagine first, then body part
8. **Stricter deep learning** — early stopping uses validation subjects carved out of the *training* fold (v6 early-stopped on the test fold, which is slightly optimistic)
9. **Feature cache** — slow feature extraction is saved to `v7_cache/` so re-runs are fast

**New in v8** (see the v8 section near the end):

10. **Group by person, not by recording file** — Anu and kaushal each have two recordings; v6/v7 treated them as
    four different subjects, so one person could appear in both train and test. v8 groups folds by participant folder.
11. **Subject-wise feature alignment** — every feature is z-scored per recording using that recording's own windows
    (label-free, like the raw-signal normalization above), and covariance matrices are re-centred per recording
    (Riemannian re-centring). This removes the between-person offsets that dominate cross-subject EEG.
12. **Trial-level decisions** — each 10 s trial yields 17 overlapping 2 s windows; their class probabilities are
    averaged to give one prediction per trial. All windows of a trial are always in the same test fold.
13. **Fixed ensemble** — Logistic Regression + shrinkage LDA + Extra Trees + HistGB, chosen up front and used
    unchanged for all three tasks (no per-task model picking on test scores).


In [ ]:
%pip install mne pyriemann pywavelets

## Imports

In [ ]:
import os, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from scipy.signal import welch, coherence, stft
from scipy.stats import kurtosis, skew

from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.feature_selection import SelectKBest, f_classif
from mne.decoding import CSP

from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from pyriemann.classification import MDM

import pywt

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')
print('ML imports OK')

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense, Dropout, Input, Conv1D, Conv2D, MaxPooling1D, BatchNormalization,
    Flatten, LSTM, AveragePooling2D, DepthwiseConv2D, SeparableConv2D, Activation
)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.constraints import max_norm

gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print(f'TensorFlow {tf.__version__} — {"GPU" if gpus else "CPU"}')

## Configuration

In [ ]:
def _has_edf(path):
    if not os.path.isdir(path): return False
    for _, _, files in os.walk(path):
        if any(f.endswith('.edf') for f in files): return True
    return False

_candidates = [
    os.path.join(os.getcwd(), 'Grp18_protocal'),
    os.getcwd(),
    r'C:\Users\ravul\OneDrive\Desktop\Project\BCI\Grp18_protocal',
    r'C:\Users\tanay\Desktop\Grp18_protocal',
]
BASE_PATH = next((p for p in _candidates if _has_edf(p)), None)
assert BASE_PATH, 'No folder with .edf files found — set BASE_PATH manually'
print(f'Data path: {BASE_PATH}')

EMOTIV_CHANNELS = ['AF3','AF4','F3','F4','F7','F8','FC5','FC6','T7','T8','P7','P8','O1','O2']
TASK_LABELS = [
    'RightArm_Action','RightArm_Imagine','LeftArm_Action','LeftArm_Imagine',
    'RightLeg_Action','RightLeg_Imagine','LeftLeg_Action','LeftLeg_Imagine',
    'RightWrist_Action','RightWrist_Imagine','LeftWrist_Action','LeftWrist_Imagine'
]
BODY_PARTS = ['RightArm','LeftArm','RightLeg','LeftLeg','RightWrist','LeftWrist']

HEMI_PAIRS = [('AF3','AF4'), ('F3','F4'), ('F7','F8'), ('FC5','FC6'), ('T7','T8'), ('P7','P8'), ('O1','O2')]
HEMI_A = [EMOTIV_CHANNELS.index(a) for a, _ in HEMI_PAIRS]
HEMI_B = [EMOTIV_CHANNELS.index(b) for _, b in HEMI_PAIRS]

BANDS = {'theta': (4, 8), 'mu': (8, 12), 'beta': (13, 30), 'gamma': (30, 45)}
FBCSP_BANDS = [(4, 8), (8, 12), (12, 16), (16, 20), (20, 24), (24, 30), (30, 40)]
CWT_FREQS = np.array([5, 7, 10, 12, 15, 20, 25, 30, 35, 40], dtype=float)

START_OFFSET = 15; RELAX_TIME = 5; TASK_TIME = 10; SFREQ = 128
WINDOW_SEC = 2; STRIDE_SEC = 0.5
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Config set.')

## Data Loading

In [ ]:
person_ids = []; X_segments = []; y_binary = []; y_multiclass = []; y_bodypart = []; subject_ids = []
participant_id = 0

for root, dirs, files in os.walk(BASE_PATH):
    if 'segmented_tasks' in root: continue
    for file in sorted(files):
        if not file.endswith('.edf'): continue
        file_path = os.path.join(root, file)
        if os.path.getsize(file_path) < 500000: continue

        raw = mne.io.read_raw_edf(file_path, preload=True)
        available = [ch for ch in EMOTIV_CHANNELS if ch in raw.ch_names]
        if len(available) < 14: continue
        participant_id += 1
        raw.pick(EMOTIV_CHANNELS)
        raw.filter(4.0, 45.0, fir_design='firwin')
        sfreq = int(raw.info['sfreq'])
        samples_per_trial = TASK_TIME * sfreq
        start = START_OFFSET

        for task_id in range(len(TASK_LABELS)):
            s0 = int(start * sfreq); s1 = s0 + samples_per_trial
            if s1 > raw.n_times: break
            seg = raw.get_data(start=s0, stop=s1)
            if seg.shape[1] == samples_per_trial:
                X_segments.append(seg)
                y_binary.append(task_id % 2)
                y_multiclass.append(task_id)
                y_bodypart.append(task_id // 2)
                subject_ids.append(participant_id)
                person_ids.append(os.path.relpath(root, BASE_PATH).split(os.sep)[0])
            start += TASK_TIME + RELAX_TIME

        if participant_id % 10 == 0: print(f'  {participant_id} participants...')

print(f'Total: {participant_id} participants, {len(X_segments)} trials')

## Artifact Rejection

In [ ]:
X = np.nan_to_num(np.stack(X_segments))
y_bin = np.array(y_binary); y_multi = np.array(y_multiclass)
y_body = np.array(y_bodypart); subj = np.array(subject_ids)

good = [i for i in range(len(X))
        if np.std(X[i]) > 0
        and np.max(np.abs(X[i])) < 150e-6
        and np.mean(np.abs(kurtosis(X[i], axis=1))) < 10]
rejected = len(X) - len(good)
X = X[good]; y_bin = y_bin[good]; y_multi = y_multi[good]; y_body = y_body[good]; subj = subj[good]
person = np.array(person_ids)[good]

print(f'Rejected {rejected}, clean: {X.shape}')
print(f'Binary: {np.bincount(y_bin)}, 12-class: {np.bincount(y_multi)}')
print(f'Recordings: {len(np.unique(subj))}, people: {len(np.unique(person))}')

## Per-Subject Z-Normalization (NEW)

Each person has different scalp impedance, electrode contact and baseline amplitude.
Each channel is z-scored using **that subject's own** mean/std over all their trials.
No labels are used, so this is not leakage — a real BCI can do the same with a short
unlabeled calibration recording from a new user.

In [ ]:
for s in np.unique(subj):
    m = subj == s
    mu = X[m].mean(axis=(0, 2), keepdims=True)
    sd = X[m].std(axis=(0, 2), keepdims=True) + 1e-12
    X[m] = (X[m] - mu) / sd

print(f'Normalized {len(np.unique(subj))} subjects — global mean {X.mean():.4f}, std {X.std():.4f}')

## Sliding Window Augmentation

In [ ]:
win_samp = int(WINDOW_SEC * SFREQ); stride_samp = int(STRIDE_SEC * SFREQ)

win_index = []
for i in range(len(X)):
    for s in range(0, X.shape[2] - win_samp + 1, stride_samp):
        win_index.append((i, s))
trial_of_win = np.array([i for i, _ in win_index])

X_win = np.array([X[i][:, s:s + win_samp] for i, s in win_index])
yw_bin = y_bin[trial_of_win]; yw_multi = y_multi[trial_of_win]
yw_body = y_body[trial_of_win]
# sw = person id (CV groups, so a person is never in both train and test); sess_w = recording id
_, person_code = np.unique(person, return_inverse=True)
sw = person_code[trial_of_win]; sess_w = subj[trial_of_win]

print(f'Windows: {X_win.shape} ({X_win.shape[0] / X.shape[0]:.1f}x augmentation)')
print(f'People: {len(np.unique(sw))}, recordings: {len(np.unique(sess_w))}')

## FBCSP Sub-Band Filtering (NEW)

Filter the full 10 s trials into 7 sub-bands **once**, then cut the same windows.
Filtering is label-free, so doing it before cross-validation is safe — only the CSP
spatial filters (which use labels) are learned inside each fold.
Filtering the long trials instead of 2 s windows also avoids filter edge artifacts.

In [ ]:
X_bands = {}
for lo, hi in FBCSP_BANDS:
    Xf = mne.filter.filter_data(X.astype(np.float64), SFREQ, lo, hi, verbose=False)
    X_bands[(lo, hi)] = np.array([Xf[i][:, s:s + win_samp] for i, s in win_index], dtype=np.float32)
    print(f'  {lo:>2}-{hi:<2} Hz done')
del Xf
print(f'FBCSP: {len(FBCSP_BANDS)} bands x {X_win.shape}')

## Feature Extraction

All features below are computed per window without labels (no leakage).
Results are cached in `v7_cache/` — **delete that folder if you change preprocessing
or add new subjects with the same window count.**

In [ ]:
CACHE_DIR = 'v7_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

def cached(name, fn):
    path = os.path.join(CACHE_DIR, f'{name}_{X_win.shape[0]}.npy')
    if os.path.exists(path):
        arr = np.load(path)
        print(f'{name}: loaded {arr.shape} from cache')
        return arr
    t0 = time.time()
    arr = np.nan_to_num(fn(), nan=0.0, posinf=0.0, neginf=0.0)
    np.save(path, arr)
    print(f'{name}: {arr.shape} in {time.time() - t0:.0f}s')
    return arr

### Original v6 features (band power, statistical, fractal, wavelet)

In [ ]:
def higuchi_fd(x, kmax=10):
    n = len(x); lk = []
    for k in range(1, kmax + 1):
        lm = []
        for m in range(1, k + 1):
            count = int(np.floor((n - m) / k))
            if count < 1: continue
            idx = m - 1 + np.arange(count + 1) * k
            ll = np.sum(np.abs(np.diff(x[idx]))) * (n - 1) / (k * count * k)
            lm.append(ll)
        if lm: lk.append(np.mean(lm))
    lk = np.array(lk); lk = lk[lk > 0]
    if len(lk) < 2: return 0.0
    return np.polyfit(np.log(1.0 / np.arange(1, len(lk) + 1)), np.log(lk), 1)[0]

def petrosian_fd(x):
    n = len(x); d = np.diff(x)
    n_delta = np.sum(d[:-1] * d[1:] < 0)
    if n_delta == 0: return 0.0
    return np.log10(n) / (np.log10(n) + np.log10(n / (n + 0.4 * n_delta)))

def hurst_exponent(x):
    n = len(x)
    max_k = min(n // 4, 50)
    if max_k < 4: return 0.5
    rs_list = []; ns_list = []
    for k in range(4, max_k + 1):
        seg = x[:k * (n // k)].reshape(n // k, k)
        dev = np.cumsum(seg - seg.mean(axis=1, keepdims=True), axis=1)
        sd = seg.std(axis=1)
        ok = sd > 0
        if ok.any():
            rs_list.append(np.mean((dev.max(axis=1) - dev.min(axis=1))[ok] / sd[ok])); ns_list.append(k)
    if len(rs_list) < 2: return 0.5
    return float(np.clip(np.polyfit(np.log(ns_list), np.log(rs_list), 1)[0], 0, 1))

def extract_v6_features(Xw):
    feats = []
    for i in range(len(Xw)):
        row = []
        for ch in range(Xw.shape[1]):
            sig = Xw[i, ch]
            freqs, psd = welch(sig, fs=SFREQ, nperseg=min(128, len(sig)))
            total = psd.mean() + 1e-10
            bp = {b: psd[(freqs >= lo) & (freqs <= hi)].mean() for b, (lo, hi) in BANDS.items()}
            row += [np.log1p(bp[b]) for b in BANDS]
            row += [bp[b] / total for b in BANDS]
            row += [bp['beta'] / (bp['mu'] + 1e-10), bp['mu'] / (bp['theta'] + 1e-10),
                    (bp['mu'] + bp['beta']) / (bp['theta'] + bp['gamma'] + 1e-10)]
            d1 = np.diff(sig); d2 = np.diff(d1)
            mob = d1.std() / (sig.std() + 1e-10)
            row += [sig.var(), kurtosis(sig), skew(sig), np.abs(sig).mean(), mob,
                    (d2.std() / (d1.std() + 1e-10)) / (mob + 1e-10)]
            row += [higuchi_fd(sig, kmax=8), petrosian_fd(sig), hurst_exponent(sig)]
            row += [np.log1p(np.sum(c ** 2)) for c in pywt.wavedec(sig, 'db4', level=3)]
        feats.append(row)
        if (i + 1) % 1000 == 0: print(f'  {i + 1}/{len(Xw)}...')
    return np.array(feats)

X_feat = cached('v6_features', lambda: extract_v6_features(X_win))

### Spectrogram features (NEW — STFT)

STFT with 0.5 s segments (50% overlap) gives power in each band at several time points
inside the 2 s window. Static band power averages these away; ERD/ERS is a *change* over time.
Per channel: 4 bands x (time bins + temporal variance).

In [ ]:
STFT_NPERSEG = 64; STFT_NOVERLAP = 32

def extract_spectrogram_features(Xw, batch=1000):
    out = []
    for b in range(0, len(Xw), batch):
        f, t, Z = stft(Xw[b:b + batch], fs=SFREQ, nperseg=STFT_NPERSEG, noverlap=STFT_NOVERLAP, axis=-1)
        P = np.abs(Z) ** 2                      # (N, C, F, T)
        per_band = []
        for lo, hi in BANDS.values():
            bp = P[:, :, (f >= lo) & (f <= hi), :].mean(axis=2)   # (N, C, T)
            per_band.append(np.concatenate([np.log1p(bp), np.log1p(bp.var(axis=-1, keepdims=True))], axis=-1))
        out.append(np.stack(per_band, axis=2).reshape(len(P), -1))
    return np.vstack(out)

N_STFT_T = len(stft(np.zeros(win_samp), fs=SFREQ, nperseg=STFT_NPERSEG, noverlap=STFT_NOVERLAP)[1])
X_spectro = cached('spectrogram', lambda: extract_spectrogram_features(X_win))
print(f'STFT time bins per window: {N_STFT_T}')

### Scalogram features (NEW — CWT, Morlet wavelet)

Morlet scales are chosen so the wavelet centre frequencies are exactly 5–40 Hz.
Per channel: energy at 10 frequencies x 4 time bins, high/low frequency energy ratio,
and temporal variance at each frequency.

In [ ]:
CWT_SCALES = pywt.central_frequency('morl') * SFREQ / CWT_FREQS
_, _cwt_f = pywt.cwt(np.zeros(win_samp), CWT_SCALES, 'morl', sampling_period=1.0 / SFREQ)
print('CWT centre frequencies (Hz):', np.round(_cwt_f, 1))

def extract_scalogram_features(Xw, batch=300, n_bins=4):
    out = []
    for b in range(0, len(Xw), batch):
        coef, _ = pywt.cwt(Xw[b:b + batch], CWT_SCALES, 'morl', sampling_period=1.0 / SFREQ, axis=-1)
        P = np.moveaxis(np.abs(coef) ** 2, 0, 2)          # (N, C, S, T)
        n, c, s, t = P.shape
        binned = np.log1p(P[..., :t // n_bins * n_bins].reshape(n, c, s, n_bins, -1).mean(-1)).reshape(n, c, s * n_bins)
        low, high = P[:, :, :5].mean(axis=(2, 3)), P[:, :, 5:].mean(axis=(2, 3))
        ratio = (high / (low + 1e-10))[..., None]
        var = np.log1p(P.var(axis=-1))
        out.append(np.concatenate([binned, ratio, var], axis=2).reshape(n, -1))
    return np.vstack(out)

X_scalo = cached('scalogram', lambda: extract_scalogram_features(X_win))

### Connectivity + asymmetry features (NEW)

Imagining a right-side movement desynchronizes the *left* hemisphere (and vice versa).
For each left/right electrode pair: coherence in 4 bands + log power ratio (left/right) in 4 bands.

In [ ]:
def extract_connectivity_features(Xw):
    f, coh = coherence(Xw[:, HEMI_A], Xw[:, HEMI_B], fs=SFREQ, nperseg=128, axis=-1)   # (N, P, F)
    fw, psd = welch(Xw, fs=SFREQ, nperseg=128, axis=-1)                                # (N, C, F)
    coh_f, asym_f = [], []
    for lo, hi in BANDS.values():
        coh_f.append(coh[..., (f >= lo) & (f <= hi)].mean(-1))
        m = (fw >= lo) & (fw <= hi)
        pa = psd[:, HEMI_A][..., m].mean(-1); pb = psd[:, HEMI_B][..., m].mean(-1)
        asym_f.append(np.log((pa + 1e-10) / (pb + 1e-10)))
    return np.concatenate([np.stack(coh_f, -1), np.stack(asym_f, -1)], axis=2).reshape(len(Xw), -1)

X_conn = cached('connectivity', lambda: extract_connectivity_features(X_win))

## Combined Feature Set + Names

In [ ]:
X_all = np.hstack([X_feat, X_spectro, X_scalo, X_conn])

FEAT_TYPES_V6 = [
    'log_theta','log_mu','log_beta','log_gamma','rel_theta','rel_mu','rel_beta','rel_gamma',
    'beta_mu_ratio','mu_theta_ratio','sensorimotor_ratio',
    'variance','kurtosis','skewness','MAV','hjorth_mobility','hjorth_complexity',
    'higuchi_fd','petrosian_fd','hurst_exponent',
    'wavelet_L0','wavelet_L1','wavelet_L2','wavelet_L3'
]
FEAT_NAMES = [f'{ch}_{ft}' for ch in EMOTIV_CHANNELS for ft in FEAT_TYPES_V6]
FEAT_NAMES += [f'{ch}_stft_{b}_{"t" + str(t) if t < N_STFT_T else "var"}'
               for ch in EMOTIV_CHANNELS for b in BANDS for t in range(N_STFT_T + 1)]
for ch in EMOTIV_CHANNELS:
    FEAT_NAMES += [f'{ch}_cwt_{int(fq)}Hz_t{t}' for fq in CWT_FREQS for t in range(4)]
    FEAT_NAMES += [f'{ch}_cwt_hi_lo_ratio']
    FEAT_NAMES += [f'{ch}_cwt_{int(fq)}Hz_var' for fq in CWT_FREQS]
FEAT_NAMES += [f'{a}-{b}_{kind}_{band}' for a, b in HEMI_PAIRS for kind in ('coh', 'asym') for band in BANDS]
assert len(FEAT_NAMES) == X_all.shape[1], (len(FEAT_NAMES), X_all.shape[1])

FEAT_FAMILY = np.array(['Original v6'] * X_feat.shape[1] + ['Spectrogram'] * X_spectro.shape[1]
                       + ['Scalogram'] * X_scalo.shape[1] + ['Connectivity'] * X_conn.shape[1])

for fam in ['Original v6', 'Spectrogram', 'Scalogram', 'Connectivity']:
    print(f'{fam:>14}: {np.sum(FEAT_FAMILY == fam)} features')
print(f'{"Total":>14}: {X_all.shape[1]} features')

## Leak-Free Helpers

In [ ]:
def apply_fbcsp(tr, te, y_tr, n_components=4):
    f_tr, f_te = [], []
    for band, Xb in X_bands.items():
        csp = CSP(n_components=n_components, reg='ledoit_wolf', log=True)
        f_tr.append(csp.fit_transform(Xb[tr].astype(np.float64), y_tr))
        f_te.append(csp.transform(Xb[te].astype(np.float64)))
    return np.hstack(f_tr), np.hstack(f_te)

def apply_riemann(tr, te):
    cov = Covariances(estimator='lwf')
    C_tr = cov.fit_transform(X_win[tr]); C_te = cov.transform(X_win[te])
    ts = TangentSpace()
    return C_tr, C_te, ts.fit_transform(C_tr), ts.transform(C_te)

def scale(a, b):
    sc = StandardScaler()
    return sc.fit_transform(a), sc.transform(b)

MODELS = {
    'SVM': lambda: SVC(C=10, kernel='rbf', gamma='scale'),
    'RF': lambda: RandomForestClassifier(n_estimators=300, max_depth=15, random_state=RANDOM_STATE, n_jobs=-1),
    'HistGB': lambda: HistGradientBoostingClassifier(max_iter=200, max_depth=5, learning_rate=0.05, random_state=RANDOM_STATE),
    'Naive Bayes': lambda: GaussianNB(),
    'AdaBoost': lambda: AdaBoostClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'LDA (shrinkage)': lambda: LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'),
}

# Multi-class: drop the slowest/weakest (AdaBoost, Naive Bayes) and use fewer boosting rounds
MULTI_MODELS = {
    'SVM': MODELS['SVM'],
    'RF': MODELS['RF'],
    'HistGB': lambda: HistGradientBoostingClassifier(max_iter=100, max_depth=5, learning_rate=0.1, random_state=RANDOM_STATE),
    'LDA (shrinkage)': MODELS['LDA (shrinkage)'],
}

def run_ml(y, n_splits, k_best, n_csp, title, models=MODELS):
    gkf = GroupKFold(n_splits=n_splits)
    res = {}
    print(f'=== {title} ({n_splits}-fold GroupKFold) ===')
    for fold, (tr, te) in enumerate(gkf.split(X_win, y, groups=sw)):
        t0 = time.time()
        sel = SelectKBest(f_classif, k=k_best).fit(X_all[tr], y[tr])
        C_tr, C_te, R_tr, R_te = apply_riemann(tr, te)
        F_tr, F_te = apply_fbcsp(tr, te, y[tr], n_components=n_csp)

        feat_sets = {
            'FBCSP': scale(F_tr, F_te),
            'Enhanced All': scale(X_all[tr], X_all[te]),
            f'Selected (Top {k_best})': scale(sel.transform(X_all[tr]), sel.transform(X_all[te])),
            'Riemann': scale(R_tr, R_te),
            'Mega v7 (FBCSP+All+Riem)': scale(np.hstack([F_tr, X_all[tr], R_tr]), np.hstack([F_te, X_all[te], R_te])),
        }
        for fname, (Xtr, Xte) in feat_sets.items():
            for mname, mk in models.items():
                clf = mk().fit(Xtr, y[tr])
                res.setdefault(f'{fname} + {mname}', []).append(accuracy_score(y[te], clf.predict(Xte)))
        res.setdefault('Riemann MDM', []).append(accuracy_score(y[te], MDM().fit(C_tr, y[tr]).predict(C_te)))
        print(f'  Fold {fold + 1}: train {len(np.unique(sw[tr]))} subj / test {len(np.unique(sw[te]))} subj — {time.time() - t0:.0f}s')
    return res

def summarize(res, chance, v6_best, top=15):
    mean = {k: np.mean(v) for k, v in res.items()}
    std = {k: np.std(v) for k, v in res.items()}
    for k, v in sorted(mean.items(), key=lambda x: -x[1])[:top]:
        print(f'  {k:>45}: {v:.4f} (+/- {std[k]:.4f})')
    print(f'  {"Chance":>45}: {chance:.4f}')
    print(f'  {"v6 best":>45}: {v6_best:.4f}')
    return mean, std

## Binary Classification — Action vs Imagine (ML)

In [ ]:
res_bin = run_ml(yw_bin, n_splits=5, k_best=150, n_csp=4, title='Binary: Action vs Imagine')
results_bin_mean, results_bin_std = summarize(res_bin, 0.5, 0.6549)

## 12-Class Classification — All Tasks (ML)

In [ ]:
res_12 = run_ml(yw_multi, n_splits=3, k_best=200, n_csp=4, title='12-Class: All Tasks', models=MULTI_MODELS)
results_12_mean, results_12_std = summarize(res_12, 1 / 12, 0.1711)

## 6-Class Classification — Body Parts (ML)

In [ ]:
res_6 = run_ml(yw_body, n_splits=3, k_best=200, n_csp=4, title='6-Class: Body Parts', models=MULTI_MODELS)
results_6_mean, results_6_std = summarize(res_6, 1 / 6, 0.2459)

## Hierarchical 12-Class (NEW)

1. **Stage 1:** Action vs Imagine (our strongest classifier)
2. **Stage 2:** Body part (6-class), with a separate model trained for Action trials and for Imagine trials

Both stages are trained only on training subjects in each fold.

In [ ]:
gkf3 = GroupKFold(n_splits=3)
hier_scores = []

for fold, (tr, te) in enumerate(gkf3.split(X_win, yw_multi, groups=sw)):
    C_tr, C_te, R_tr, R_te = apply_riemann(tr, te)
    F_tr, F_te = apply_fbcsp(tr, te, yw_bin[tr])
    S1_tr, S1_te = scale(np.hstack([F_tr, X_all[tr], R_tr]), np.hstack([F_te, X_all[te], R_te]))
    clf_bin = RandomForestClassifier(n_estimators=300, max_depth=15, random_state=RANDOM_STATE, n_jobs=-1)
    pred_bin = clf_bin.fit(S1_tr, yw_bin[tr]).predict(S1_te)

    final_pred = np.zeros(len(te), dtype=int)
    for cond in (0, 1):
        trm = yw_bin[tr] == cond
        sc = StandardScaler().fit(np.hstack([X_all[tr][trm], R_tr[trm]]))
        clf_body = SVC(C=10, kernel='rbf', gamma='scale')
        clf_body.fit(sc.transform(np.hstack([X_all[tr][trm], R_tr[trm]])), yw_body[tr][trm])
        tem = pred_bin == cond
        if tem.any():
            body_pred = clf_body.predict(sc.transform(np.hstack([X_all[te][tem], R_te[tem]])))
            final_pred[tem] = body_pred * 2 + cond

    acc = accuracy_score(yw_multi[te], final_pred)
    hier_scores.append(acc)
    print(f'Fold {fold + 1}: stage-1 binary {accuracy_score(yw_bin[te], pred_bin):.4f} -> hierarchical 12-class {acc:.4f}')

results_12_mean['Hierarchical (Binary -> 6-class)'] = np.mean(hier_scores)
print(f'\nHierarchical 12-class: {np.mean(hier_scores):.4f}  (v6 direct best: 0.1711)')

## Deep Learning (GroupKFold, strict validation)

For each test fold, ~15% of the **training subjects** are held out as a validation set
for early stopping. The test subjects are never seen until the final evaluation.

In [ ]:
def inner_val_split(tr, val_frac=0.15):
    gss = GroupShuffleSplit(n_splits=1, test_size=val_frac, random_state=RANDOM_STATE)
    i_tr, i_va = next(gss.split(tr, groups=sw[tr]))
    return tr[i_tr], tr[i_va]

def run_dl(build_fn, X_in, y, n_classes, n_splits=3, epochs=100, patience=15, name='model'):
    scores = []
    loss = 'binary_crossentropy' if n_classes == 2 else 'categorical_crossentropy'
    enc = (lambda a: a) if n_classes == 2 else (lambda a: to_categorical(a, n_classes))
    for fold, (tr, te) in enumerate(GroupKFold(n_splits=n_splits).split(X_in, y, groups=sw)):
        tr_in, va = inner_val_split(tr)
        mu, sd = X_in[tr_in].mean(), X_in[tr_in].std() + 1e-8
        norm = lambda a: (a - mu) / sd
        tf.keras.backend.clear_session()
        tf.random.set_seed(RANDOM_STATE)
        m = build_fn()
        m.compile(optimizer='adam', loss=loss, metrics=['accuracy'])
        m.fit(norm(X_in[tr_in]), enc(y[tr_in]), epochs=epochs, batch_size=64,
              validation_data=(norm(X_in[va]), enc(y[va])),
              callbacks=[EarlyStopping(patience=patience, restore_best_weights=True)], verbose=0)
        _, acc = m.evaluate(norm(X_in[te]), enc(y[te]), verbose=0)
        scores.append(acc)
        print(f'  {name} fold {fold + 1}: {acc:.4f}')
    print(f'  {name} mean: {np.mean(scores):.4f}')
    return np.mean(scores)

def build_eegnet(n_classes, n_ch=14, n_t=256, F1=8, D=2, F2=16, drop=0.5):
    inp = Input(shape=(n_ch, n_t, 1))
    x = Conv2D(F1, (1, 64), padding='same', use_bias=False)(inp)
    x = BatchNormalization()(x)
    x = DepthwiseConv2D((n_ch, 1), use_bias=False, depth_multiplier=D, depthwise_constraint=max_norm(1.))(x)
    x = BatchNormalization()(x); x = Activation('elu')(x)
    x = AveragePooling2D((1, 4))(x); x = Dropout(drop)(x)
    x = SeparableConv2D(F2, (1, 16), padding='same', use_bias=False)(x)
    x = BatchNormalization()(x); x = Activation('elu')(x)
    x = AveragePooling2D((1, 8))(x); x = Dropout(drop)(x)
    x = Flatten()(x)
    out = (Dense(1, activation='sigmoid', kernel_constraint=max_norm(0.25))(x) if n_classes == 2
           else Dense(n_classes, activation='softmax', kernel_constraint=max_norm(0.25))(x))
    return Model(inp, out)

def build_cnn_lstm(n_classes, n_t=256, n_ch=14):
    inp = Input(shape=(n_t, n_ch))
    x = Conv1D(32, 32, strides=2, activation='relu', padding='same')(inp)
    x = BatchNormalization()(x); x = MaxPooling1D(2)(x); x = Dropout(0.3)(x)
    x = Conv1D(64, 16, activation='relu', padding='same')(x)
    x = BatchNormalization()(x); x = MaxPooling1D(2)(x); x = Dropout(0.3)(x)
    x = LSTM(64, return_sequences=True)(x); x = Dropout(0.3)(x)
    x = LSTM(32)(x); x = Dropout(0.3)(x)
    x = Dense(32, activation='relu')(x); x = Dropout(0.2)(x)
    out = Dense(1, activation='sigmoid')(x) if n_classes == 2 else Dense(n_classes, activation='softmax')(x)
    return Model(inp, out)

X_eeg = X_win[..., np.newaxis].astype(np.float32)          # EEGNet: (N, channels, time, 1)
X_seq = np.transpose(X_win, (0, 2, 1)).astype(np.float32)  # CNN+LSTM: (N, time, channels)
build_eegnet(2).summary()

In [ ]:
print('=== Binary DL ===')
results_bin_mean['EEGNet'] = run_dl(lambda: build_eegnet(2), X_eeg, yw_bin, 2, name='EEGNet')
results_bin_mean['CNN+LSTM'] = run_dl(lambda: build_cnn_lstm(2), X_seq, yw_bin, 2, epochs=50, patience=10, name='CNN+LSTM')

print('\n=== 12-Class DL ===')
results_12_mean['EEGNet'] = run_dl(lambda: build_eegnet(12), X_eeg, yw_multi, 12, name='EEGNet')

print('\n=== 6-Class DL ===')
results_6_mean['EEGNet'] = run_dl(lambda: build_eegnet(6), X_eeg, yw_body, 6, name='EEGNet')

## Spectrogram Visualization (STFT)

In [ ]:
VIS_CH = ['AF3', 'F3', 'FC5', 'T8']
vis_idx = [EMOTIV_CHANNELS.index(c) for c in VIS_CH]
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for row, (label, name) in enumerate([(0, 'Action'), (1, 'Imagine')]):
    idx = np.where(yw_bin == label)[0]
    f, t, Z = stft(X_win[idx][:, vis_idx], fs=SFREQ, nperseg=64, noverlap=56, axis=-1)
    P = (np.abs(Z) ** 2).mean(axis=0)                     # (C, F, T)
    fm = f <= 45
    for col, ch in enumerate(VIS_CH):
        ax = axes[row, col]
        ax.pcolormesh(t, f[fm], 10 * np.log10(P[col][fm] + 1e-12), shading='gouraud', cmap='jet')
        for y in (8, 12, 30): ax.axhline(y, color='w', ls='--', alpha=0.5)
        ax.set_title(f'{ch} — {name}'); ax.set_xlabel('Time (s)'); ax.set_ylabel('Hz')
plt.suptitle('Average Spectrogram (STFT): Action vs Imagine', fontsize=14)
plt.tight_layout(); plt.show()

## Scalogram Visualization (CWT, Morlet)

In [ ]:
vis_freqs = np.linspace(4, 45, 40)
vis_scales = pywt.central_frequency('morl') * SFREQ / vis_freqs
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
t_axis = np.arange(win_samp) / SFREQ
for row, (label, name) in enumerate([(0, 'Action'), (1, 'Imagine')]):
    idx = np.where(yw_bin == label)[0][:400]
    coef, _ = pywt.cwt(X_win[idx][:, vis_idx], vis_scales, 'morl', sampling_period=1.0 / SFREQ, axis=-1)
    P = (np.abs(coef) ** 2).mean(axis=1)                  # (S, C, T)
    for col, ch in enumerate(VIS_CH):
        ax = axes[row, col]
        ax.pcolormesh(t_axis, vis_freqs, 10 * np.log10(P[:, col] + 1e-12), shading='gouraud', cmap='jet')
        for y in (8, 12, 30): ax.axhline(y, color='w', ls='--', alpha=0.5)
        ax.set_title(f'{ch} — {name}'); ax.set_xlabel('Time (s)'); ax.set_ylabel('Hz')
plt.suptitle('Average Scalogram (CWT Morlet): Action vs Imagine', fontsize=14)
plt.tight_layout(); plt.show()

## Feature Importance (ANOVA averaged over training folds)

In [ ]:
scores_anova = np.zeros(X_all.shape[1])
for tr, _ in GroupKFold(n_splits=5).split(X_all, yw_bin, groups=sw):
    scores_anova += np.nan_to_num(f_classif(X_all[tr], yw_bin[tr])[0])
scores_anova /= 5

print('Top 30 features (Action vs Imagine):')
for rank, i in enumerate(np.argsort(scores_anova)[::-1][:30]):
    print(f'  {rank + 1:2d}. {FEAT_NAMES[i]:>32} [{FEAT_FAMILY[i]}]  F={scores_anova[i]:.1f}')

fams = ['Original v6', 'Spectrogram', 'Scalogram', 'Connectivity']
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
top30 = np.argsort(scores_anova)[-30:]
fam_color = dict(zip(fams, ['#607D8B', '#2196F3', '#FF9800', '#4CAF50']))
axes[0].barh(range(30), scores_anova[top30], color=[fam_color[FEAT_FAMILY[i]] for i in top30])
axes[0].set_yticks(range(30)); axes[0].set_yticklabels([FEAT_NAMES[i] for i in top30], fontsize=7)
axes[0].set_xlabel('ANOVA F-score'); axes[0].set_title('Top 30 Features (colored by family)')
axes[0].legend(handles=[plt.Rectangle((0, 0), 1, 1, color=fam_color[f]) for f in fams], labels=fams)
axes[1].bar(fams, [np.median(scores_anova[FEAT_FAMILY == f]) for f in fams], color=[fam_color[f] for f in fams])
axes[1].set_ylabel('Median F-score'); axes[1].set_title('Discriminative Power by Feature Family')
plt.tight_layout(); plt.show()

## Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
for ax, (title, res, chance, v6) in zip(axes, [
    ('Binary', results_bin_mean, 0.5, 0.6549),
    ('12-Class', results_12_mean, 1 / 12, 0.1711),
    ('6-Class', results_6_mean, 1 / 6, 0.2459),
]):
    top = sorted(res.items(), key=lambda x: -x[1])[:12][::-1]
    ax.barh(range(len(top)), [v for _, v in top], color='#00A896')
    ax.set_yticks(range(len(top))); ax.set_yticklabels([k for k, _ in top], fontsize=7)
    ax.axvline(chance, color='red', ls='--', label=f'Chance ({chance:.1%})')
    ax.axvline(v6, color='orange', ls='--', label=f'v6 best ({v6:.1%})')
    ax.set_title(f'{title} — Top 12 (v7)'); ax.set_xlabel('Accuracy'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Confusion Matrices (best Mega v7 setup, one held-out fold)

In [ ]:
tr, te = next(GroupKFold(n_splits=5).split(X_win, yw_bin, groups=sw))
C_tr, C_te, R_tr, R_te = apply_riemann(tr, te)
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, y, labels, rot in [(axes[0], yw_bin, ['Action', 'Imagine'], 0), (axes[1], yw_multi, TASK_LABELS, 45)]:
    F_tr, F_te = apply_fbcsp(tr, te, y[tr])
    Xtr, Xte = scale(np.hstack([F_tr, X_all[tr], R_tr]), np.hstack([F_te, X_all[te], R_te]))
    pred = SVC(C=10, kernel='rbf', gamma='scale').fit(Xtr, y[tr]).predict(Xte)
    ConfusionMatrixDisplay(confusion_matrix(y[te], pred), display_labels=labels).plot(ax=ax, cmap='Blues', xticks_rotation=rot, colorbar=False)
    ax.set_title(f'{len(labels)}-class, test subjects {sorted(np.unique(sw[te]))} — acc {accuracy_score(y[te], pred):.3f}')
plt.tight_layout(); plt.show()

## Results Summary — v6 vs v7 vs v8

## v8 — Subject-Wise Alignment + Trial-Level Ensemble

**Why cross-subject accuracy is low:** every person's EEG has a different baseline (skull thickness,
electrode contact, resting rhythm). A classifier trained on 20 people sees a new person's features shifted
as a whole and misreads them.

**Fix 1 — feature alignment (label-free).** For each recording, subtract that recording's mean feature vector
and divide by its std, computed over *all* of its windows regardless of class. No labels are used, so a new
user only needs one unlabeled calibration session. Covariance matrices are re-centred per recording
(C → M<sup>-1/2</sup> C M<sup>-1/2</sup>, M = Riemannian mean of that recording) and mapped to the tangent space at the identity.

**Fix 2 — trial-level voting.** A real BCI decides once per 10 s trial, not per 2 s window. We average the
class probabilities of the 17 windows of each trial. Window-level accuracy is reported alongside.

**Leak check:** CV groups are people; scaler and classifiers are fitted on training people only;
alignment uses no labels; all windows of a trial are in the same fold.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from pyriemann.utils.mean import mean_riemann
from pyriemann.utils.base import invsqrtm

def align_features(F, groups):
    F = F.copy()
    for g in np.unique(groups):
        m = groups == g
        F[m] = (F[m] - F[m].mean(0)) / (F[m].std(0) + 1e-8)
    return F

def recenter_covs(Cv, groups):
    Cv = Cv.copy()
    for g in np.unique(groups):
        m = groups == g
        W = invsqrtm(mean_riemann(Cv[m]))
        Cv[m] = W @ Cv[m] @ W
    return Cv

covs_all = Covariances(estimator='lwf').fit_transform(X_win)
TS_rc = TangentSpace(metric='riemann').fit(np.eye(14)[None]).transform(recenter_covs(covs_all, sess_w))
X_v8 = np.hstack([align_features(X_all, sess_w), TS_rc])
print(f'v8 feature matrix: {X_v8.shape} ({X_all.shape[1]} aligned features + {TS_rc.shape[1]} re-centred tangent-space)')

V8_MODELS = {
    'LogReg': lambda: LogisticRegression(C=0.01, max_iter=3000),
    'LDA': lambda: LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'),
    'ExtraTrees': lambda: ExtraTreesClassifier(n_estimators=500, max_features='sqrt', n_jobs=-1, random_state=RANDOM_STATE),
    'HistGB': lambda: HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=4, random_state=RANDOM_STATE),
}

def run_v8(F, y, n_splits, title):
    print(f'=== {title} ({n_splits}-fold GroupKFold by person) ===')
    res = {}
    for fold, (tr, te) in enumerate(GroupKFold(n_splits=n_splits).split(F, y, groups=sw)):
        t0 = time.time()
        sc = StandardScaler().fit(F[tr]); A, B = sc.transform(F[tr]), sc.transform(F[te])
        probs = {}
        for name, mk in V8_MODELS.items():
            clf = mk().fit(A, y[tr]); probs[name] = clf.predict_proba(B); classes = clf.classes_
        probs['Ensemble'] = np.mean(list(probs.values()), axis=0)
        tt = trial_of_win[te]; ut = np.unique(tt)
        y_trial = np.array([y[te][tt == t][0] for t in ut])
        for name, Pr in probs.items():
            P_trial = np.array([Pr[tt == t].mean(0) for t in ut])
            res.setdefault(name, {'window': [], 'trial': []})
            res[name]['window'].append(accuracy_score(y[te], classes[Pr.argmax(1)]))
            res[name]['trial'].append(accuracy_score(y_trial, classes[P_trial.argmax(1)]))
        print(f'  Fold {fold + 1}: {len(np.unique(sw[te]))} test people, {len(ut)} test trials — '
              f'ensemble window {res["Ensemble"]["window"][-1]:.3f}, trial {res["Ensemble"]["trial"][-1]:.3f} ({time.time() - t0:.0f}s)')
    for name, r in res.items():
        print(f'  {name:>12}: window {np.mean(r["window"]):.4f} ± {np.std(r["window"]):.4f} | '
              f'trial {np.mean(r["trial"]):.4f} ± {np.std(r["trial"]):.4f}')
    return res

v8_bin = run_v8(X_v8, yw_bin, 5, 'v8 Binary: Action vs Imagine')
v8_12 = run_v8(X_v8, yw_multi, 3, 'v8 12-Class')
v8_6 = run_v8(X_v8, yw_body, 3, 'v8 6-Class: Body Parts')

### Ablation — what each v8 step contributes (ensemble, trial level)
Same folds and ensemble; only the feature alignment is switched off. Shows how much comes from
trial-level voting alone vs. subject-wise alignment.

In [ ]:
X_noalign = np.hstack([X_all, TangentSpace(metric='riemann').fit_transform(covs_all)])
abl = {}
for task, y, k in [('Binary', yw_bin, 5), ('12-Class', yw_multi, 3), ('6-Class', yw_body, 3)]:
    abl[task] = run_v8(X_noalign, y, k, f'Ablation (no alignment) {task}')['Ensemble']

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
for ax, (task, v8r, chance, v6) in zip(axes, [('Binary', v8_bin, 0.5, 0.6549), ('12-Class', v8_12, 1/12, 0.1711), ('6-Class', v8_6, 1/6, 0.2459)]):
    labels = ['v6 best\n(window)', 'v8 no-align\n(trial)', 'v8 window', 'v8 trial']
    vals = [v6, np.mean(abl[task]['trial']), np.mean(v8r['Ensemble']['window']), np.mean(v8r['Ensemble']['trial'])]
    errs = [0, np.std(abl[task]['trial']), np.std(v8r['Ensemble']['window']), np.std(v8r['Ensemble']['trial'])]
    ax.bar(labels, vals, yerr=errs, capsize=5, color=['#999999', '#7FB7BE', '#02C39A', '#028090'])
    ax.axhline(chance, color='red', ls='--', label=f'Chance ({chance:.1%})')
    for j, v in enumerate(vals): ax.text(j, v + 0.005, f'{v:.3f}', ha='center', fontsize=9)
    ax.set_title(f'{task}'); ax.set_ylabel('Accuracy'); ax.legend(fontsize=8)
plt.suptitle('v6 vs v8 (GroupKFold by person, error bars = fold std)'); plt.tight_layout(); plt.show()

In [ ]:
print('=' * 80)
print('v8 LEAK-FREE PIPELINE — RESULTS')
print('=' * 80)
print(f'Dataset: {len(np.unique(sw))} people ({len(np.unique(sess_w))} recordings), {X.shape[0]} trials -> {X_win.shape[0]} windows')
print(f'Features: {X_all.shape[1]} ({X_feat.shape[1]} v6 + {X_spectro.shape[1]} STFT + {X_scalo.shape[1]} CWT + {X_conn.shape[1]} connectivity)')
print('Evaluation: GroupKFold by person — test people never seen in training, feature selection, CSP, Riemann or early stopping')

for title, res, chance, v6 in [
    ('BINARY: ACTION vs IMAGINE', results_bin_mean, 0.5, 0.6549),
    ('12-CLASS: ALL TASKS', results_12_mean, 1 / 12, 0.1711),
    ('6-CLASS: BODY PARTS', results_6_mean, 1 / 6, 0.2459),
]:
    print('\n' + '=' * 80 + f'\n{title}\n' + '=' * 80)
    best = max(res.values())
    for k, v in sorted(res.items(), key=lambda x: -x[1])[:10]:
        print(f'  {k:>45}: {v:.4f}{"  <-- BEST" if v == best else ""}')
    print(f'  {"Chance":>45}: {chance:.4f}')

print('\n' + '=' * 80)
print(f'{"":14}{"Chance":>10}{"v6 best":>10}{"v7 best":>10}{"Change":>10}')
for title, res, chance, v6 in [('Binary', results_bin_mean, 0.5, 0.6549),
                               ('12-Class', results_12_mean, 1 / 12, 0.1711),
                               ('6-Class', results_6_mean, 1 / 6, 0.2459)]:
    best = max(res.values())
    print(f'{title:<14}{chance:>10.4f}{v6:>10.4f}{best:>10.4f}{best - v6:>+10.4f}')
print('=' * 80)
print('\nv8 — subject-aligned features + fixed ensemble (mean ± fold std)')
print('=' * 80)
print(f'{"":10}{"Chance":>8}{"v6 best":>9}{"v8 window":>18}{"v8 trial":>18}{"trial gain":>12}')
for title, r, chance, v6 in [('Binary', v8_bin, 0.5, 0.6549), ('12-Class', v8_12, 1 / 12, 0.1711), ('6-Class', v8_6, 1 / 6, 0.2459)]:
    w, t = r['Ensemble']['window'], r['Ensemble']['trial']
    print(f'{title:<10}{chance:>8.4f}{v6:>9.4f}{np.mean(w):>11.4f} ±{np.std(w):.3f}{np.mean(t):>11.4f} ±{np.std(t):.3f}{np.mean(t) - v6:>+12.4f}')
print('=' * 80)
print('Note: v6 numbers are window-level; v8 trial = one decision per 10 s trial (17 windows averaged).')
